# Sentiment Analysis — Azure AI Language

This notebook uses **Azure AI Language** to perform sentiment analysis on text documents.

The service returns:
- An overall document-level sentiment label: **positive**, **neutral**, or **negative**
- Per-sentence sentiment with confidence scores
- Opinion mining — identifies the *target* and *assessment* in sentences (e.g., `food` → `delicious`)

In [1]:
%pip install azure-ai-textanalytics azure-core python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\w\repos\foundry-tools\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import os
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["AZURE_LANGUAGE_ENDPOINT"]
api_key = os.environ["AZURE_LANGUAGE_KEY"]

client = TextAnalyticsClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))
print("Language client ready.")

Language client ready.


In [3]:
# Sample documents to analyze
documents = [
    "I had a wonderful experience at the restaurant. The food was delicious and the service was excellent!",
    "The hotel room was okay, nothing special. The view was nice but the bathroom was a bit small.",
    "This product is terrible. It broke after just two days and customer support was completely unhelpful.",
    "Azure AI Language is a powerful and easy-to-use service for natural language processing.",
]

print(f"Analyzing {len(documents)} documents...")

Analyzing 4 documents...


In [4]:
# Analyze sentiment (with opinion mining enabled)
results = client.analyze_sentiment(documents=documents, show_opinion_mining=True)

for i, result in enumerate(results):
    if result.is_error:
        print(f"Document {i + 1} error: {result.error.code} - {result.error.message}")
        continue

    print(f"\n--- Document {i + 1} ---")
    print(f"Text     : {documents[i][:80]}..." if len(documents[i]) > 80 else f"Text     : {documents[i]}")
    print(f"Sentiment: {result.sentiment}")
    print(f"Scores   : positive={result.confidence_scores.positive:.2f}, "
          f"neutral={result.confidence_scores.neutral:.2f}, "
          f"negative={result.confidence_scores.negative:.2f}")

    for sentence in result.sentences:
        print(f"  Sentence: '{sentence.text[:60]}...'" if len(sentence.text) > 60 else f"  Sentence: '{sentence.text}'")
        print(f"  Sentiment: {sentence.sentiment}")

        # Opinion mining: targets and assessments
        for mined_opinion in sentence.mined_opinions:
            target = mined_opinion.target
            print(f"    Target: '{target.text}' ({target.sentiment})")
            for assessment in mined_opinion.assessments:
                print(f"      Assessment: '{assessment.text}' ({assessment.sentiment})")


--- Document 1 ---
Text     : I had a wonderful experience at the restaurant. The food was delicious and the s...
Sentiment: positive
Scores   : positive=1.00, neutral=0.00, negative=0.00
  Sentence: 'I had a wonderful experience at the restaurant. '
  Sentiment: positive
    Target: 'experience' (positive)
      Assessment: 'wonderful' (positive)
  Sentence: 'The food was delicious and the service was excellent!'
  Sentiment: positive
    Target: 'food' (positive)
      Assessment: 'delicious' (positive)
    Target: 'service' (positive)
      Assessment: 'excellent' (positive)

--- Document 2 ---
Text     : The hotel room was okay, nothing special. The view was nice but the bathroom was...
Sentiment: neutral
Scores   : positive=0.15, neutral=0.76, negative=0.08
  Sentence: 'The hotel room was okay, nothing special. '
  Sentiment: neutral
    Target: 'hotel room' (negative)
      Assessment: 'okay' (mixed)
      Assessment: 'special' (negative)
  Sentence: 'The view was nice but the b